# 11.1 Euler 方法与 Runge-Kutta 方法

初值问题 $y'=f(t,y), y(t_0)=y_0$ 的数值解通过时间网格逐步推进。本节比较显式 Euler、Heun、中点法和经典 RK4，并用全局误差观察收敛阶。

In [ ]:
import math
import pathlib
import sys

import numpy as np

ROOT = pathlib.Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = next(parent for parent in pathlib.Path.cwd().parents if (parent / 'src').exists())
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    estimate_convergence_order,
    euler_step,
    global_error,
    heun_step,
    midpoint_step,
    rk4_step,
    solve_ivp_fixed_step,
)

## 一步公式

用测试方程 $y'=y, y(0)=1$ 可以直接比较一步近似和精确值 $e^h$。

In [ ]:
def exponential_rhs(_time, state):
    return state

h = 0.1
exact = math.exp(h)
one_step = {
    'Euler': euler_step(exponential_rhs, 0.0, 1.0, h)[0],
    'Heun': heun_step(exponential_rhs, 0.0, 1.0, h)[0],
    'Midpoint': midpoint_step(exponential_rhs, 0.0, 1.0, h)[0],
    'RK4': rk4_step(exponential_rhs, 0.0, 1.0, h)[0],
}
{name: (value, abs(value - exact)) for name, value in one_step.items()}

## 固定步长求解

统一的固定步长求解器可以把不同单步公式放在同一张时间网格上比较。

In [ ]:
methods = ["euler", "heun", "midpoint", "rk4"]
summary = []
for method in methods:
    result = solve_ivp_fixed_step(exponential_rhs, (0.0, 1.0), 1.0, 0.1, method=method)
    errors = global_error(result.times, result.values, lambda time: math.exp(time))
    summary.append((method, result.values[-1, 0], errors[-1]))
summary

## 观测收敛阶

如果全局误差近似满足 $E(h)pprox C h^p$，那么 $\log E$ 对 $\log h$ 的斜率就是观测阶 $p$。

In [ ]:
step_sizes = np.array([0.2, 0.1, 0.05])
orders = {}
for method in methods:
    final_errors = []
    for step_size in step_sizes:
        result = solve_ivp_fixed_step(exponential_rhs, (0.0, 1.0), 1.0, step_size, method=method)
        final_errors.append(abs(result.values[-1, 0] - math.e))
    orders[method] = estimate_convergence_order(step_sizes, final_errors)
orders

## 向量系统

二阶方程可以转化为一阶系统。例如谐振子 $u' = v, v'=-u$ 的精确轨道是圆。

In [ ]:
def oscillator(_time, state):
    return np.array([state[1], -state[0]])

oscillator_result = solve_ivp_fixed_step(
    oscillator,
    (0.0, math.pi / 2.0),
    [1.0, 0.0],
    math.pi / 200.0,
    method="rk4",
)
oscillator_result.values[-1]

## 小结

* Euler 方法结构最简单，但全局误差通常是一阶。
* Heun 和中点法通过一次额外采样获得二阶精度。
* 经典 RK4 用四个阶段获得四阶精度，是非刚性问题的常用基准方法。
* 全局误差和观测阶实验能把公式阶数转化为可验证的数值证据。